In [12]:
# Feature Engineering
# Feature engineering transforms raw data into signals that machines and humans can act on. In AIOps — raw cpu_pct is a number. cpu_above_baseline, is_spike, server_risk_score are actionable features.

In [13]:
# Load datasets
import pandas as pd
df_metrics = pd.read_csv("server_metrics.csv")
df_tickets = pd.read_csv("incidents.csv")
df_logs = pd.read_csv("app_logs.csv")

In [14]:
# Ratio features

In [15]:
df_metrics_fe = df_metrics.drop_duplicates(
    subset=["server_id", "timestamp"]
).sort_values(["server_id", "timestamp"]).copy()

# CPU to memory ratio — high CPU + low memory = different problem than both high
df_metrics_fe["cpu_memory_ratio"] = (
    df_metrics_fe["cpu_pct"] / df_metrics_fe["memory_pct"]
).round(3)

# Disk utilization ratio — how close to full
df_metrics_fe["disk_pressure"] = (
    df_metrics_fe["disk_pct"] / 100
).round(3)

print(df_metrics_fe[["server_id", "cpu_pct", "memory_pct", 
                       "cpu_memory_ratio", "disk_pressure"]].head(10))
# AIOps use — ratio features catch imbalanced resource usage that absolute values miss.

   server_id  cpu_pct  memory_pct  cpu_memory_ratio  disk_pressure
0     srv-01     49.6        94.6             0.524          0.689
5     srv-01     82.0        43.6             1.881          0.685
10    srv-01     96.6        82.7             1.168          0.882
15    srv-01     77.6        82.4             0.942          0.533
20    srv-01     22.5        73.3             0.307          0.631
25    srv-01     53.7        85.6             0.627          0.305
30    srv-01     91.8        46.3             1.983          0.618
35    srv-01     33.8        77.0             0.439          0.909
40    srv-01     70.7        35.7             1.980          0.884
45    srv-01     39.3        96.2             0.409          0.880


In [16]:
# Lag features
# AIOps use — lag features are the foundation of time series ML models. "CPU 1 hour ago" is often the strongest predictor of "CPU now".


In [17]:
# Previous reading value — what was CPU 1 hour ago?
df_metrics_fe["cpu_lag1"] = df_metrics_fe.groupby(
    "server_id", observed=True
)["cpu_pct"].shift(1)

# 3 hours ago
df_metrics_fe["cpu_lag3"] = df_metrics_fe.groupby(
    "server_id", observed=True
)["cpu_pct"].shift(3)

# Change from 1 hour ago
df_metrics_fe["cpu_change_1h"] = (
    df_metrics_fe["cpu_pct"] - df_metrics_fe["cpu_lag1"]
).round(2)

print(df_metrics_fe[["server_id", "timestamp", "cpu_pct", 
                       "cpu_lag1", "cpu_lag3", "cpu_change_1h"]].head(15))


   server_id            timestamp  cpu_pct  cpu_lag1  cpu_lag3  cpu_change_1h
0     srv-01  2026-01-01 00:00:00     49.6       NaN       NaN            NaN
5     srv-01  2026-01-01 01:00:00     82.0      49.6       NaN           32.4
10    srv-01  2026-01-01 02:00:00     96.6      82.0       NaN           14.6
15    srv-01  2026-01-01 03:00:00     77.6      96.6      49.6          -19.0
20    srv-01  2026-01-01 04:00:00     22.5      77.6      82.0          -55.1
25    srv-01  2026-01-01 05:00:00     53.7      22.5      96.6           31.2
30    srv-01  2026-01-01 06:00:00     91.8      53.7      77.6           38.1
35    srv-01  2026-01-01 07:00:00     33.8      91.8      22.5          -58.0
40    srv-01  2026-01-01 08:00:00     70.7      33.8      53.7           36.9
45    srv-01  2026-01-01 09:00:00     39.3      70.7      91.8          -31.4
50    srv-01  2026-01-01 10:00:00     43.3      39.3      33.8            4.0
55    srv-01  2026-01-01 11:00:00     59.6      43.3      70.7  

In [18]:
# Deviation from server baseline

# AIOps use — deviation from baseline is more meaningful than raw value. srv-04 at 80% CPU might be normal if its baseline is 75%. 
# srv-01 at 80% is critical if its baseline is 50%

In [19]:
# How far is each reading from this server's overall mean?
df_metrics_fe["cpu_server_mean"] = df_metrics_fe.groupby(
    "server_id", observed=True
)["cpu_pct"].transform("mean")

df_metrics_fe["cpu_deviation"] = (
    df_metrics_fe["cpu_pct"] - df_metrics_fe["cpu_server_mean"]
).round(2)

# Normalize deviation — % above or below baseline
df_metrics_fe["cpu_deviation_pct"] = (
    df_metrics_fe["cpu_deviation"] / df_metrics_fe["cpu_server_mean"] * 100
).round(2)

print(df_metrics_fe[["server_id", "cpu_pct", "cpu_server_mean", 
                       "cpu_deviation", "cpu_deviation_pct"]].head(10))

   server_id  cpu_pct  cpu_server_mean  cpu_deviation  cpu_deviation_pct
0     srv-01     49.6            59.64         -10.04             -16.83
5     srv-01     82.0            59.64          22.36              37.49
10    srv-01     96.6            59.64          36.96              61.97
15    srv-01     77.6            59.64          17.96              30.11
20    srv-01     22.5            59.64         -37.14             -62.27
25    srv-01     53.7            59.64          -5.94              -9.96
30    srv-01     91.8            59.64          32.16              53.92
35    srv-01     33.8            59.64         -25.84             -43.33
40    srv-01     70.7            59.64          11.06              18.54
45    srv-01     39.3            59.64         -20.34             -34.10


In [20]:
# Time based features

# AIOps use — incidents during business hours vs weekends have different severity and response patterns. 
# Time features let ML models learn these patterns.

In [27]:
# Extract time components — useful for pattern detection
df_metrics_fe["timestamp"] = pd.to_datetime(df_metrics_fe["timestamp"])
df_metrics_fe["hour"] = df_metrics_fe["timestamp"].dt.hour
df_metrics_fe["day_of_week"] = df_metrics_fe["timestamp"].dt.dayofweek
df_metrics_fe["is_business_hours"] = df_metrics_fe["hour"].between(9, 18).astype(int)
df_metrics_fe["is_weekend"] = (df_metrics_fe["day_of_week"] >= 5).astype(int)

print(df_metrics_fe[["server_id", "timestamp", "hour", 
                       "day_of_week", "is_business_hours", "is_weekend"]].head(10))

   server_id           timestamp  hour  day_of_week  is_business_hours  \
0     srv-01 2026-01-01 00:00:00     0            3                  0   
5     srv-01 2026-01-01 01:00:00     1            3                  0   
10    srv-01 2026-01-01 02:00:00     2            3                  0   
15    srv-01 2026-01-01 03:00:00     3            3                  0   
20    srv-01 2026-01-01 04:00:00     4            3                  0   
25    srv-01 2026-01-01 05:00:00     5            3                  0   
30    srv-01 2026-01-01 06:00:00     6            3                  0   
35    srv-01 2026-01-01 07:00:00     7            3                  0   
40    srv-01 2026-01-01 08:00:00     8            3                  0   
45    srv-01 2026-01-01 09:00:00     9            3                  1   

    is_weekend  
0            0  
5            0  
10           0  
15           0  
20           0  
25           0  
30           0  
35           0  
40           0  
45           0 